# PATH-LOSS

Χρησιμοποιήστε το dataset paper.xlsx. H τελευταία στήλη περιέχει την απώλεια διαδρομής σε dB (path loss).  A. Εκπαιδεύστε Decision Tree Regressor (fitrtree). 

Πειραματιστείτε με την παράμετρο MaxDepth, Υπολογίστε  MAE,RMSE, MAPE στο test set.

- Το dataset αποτελείται από 18.898 γραμμές (δείγματα) και 5 στήλες (χαρακτηριστικά μαζί με τον στόχο).

- Το αρχείο δεν περιέχει ονόματα στηλών (headers) στην πρώτη γραμμή, αλλά κατευθείαν αριθμητικά δεδομένα.

- Όλα τα δεδομένα είναι αριθμητικά. Οι πρώτες 3 στήλες είναι ακέραιοι αριθμοί (int64), ενώ οι τελευταίες 2 είναι δεκαδικοί (float64).

- Απουσιάζουσες Τιμές (Missing values): Δεν υπάρχει κανένα κενό ή λείπον δεδομένο στο αρχείο (0 null values), γεγονός που είναι ιδανικό και σημαίνει ότι δεν χρειάζεται να κάνεις συμπλήρωση κενών (imputation) πριν την εκπαίδευση κάποιου μοντέλου.

Αφού δεν υπάρχουν ονόματα, τις αναφέρω με τη σειρά τους (Στήλη 1 έως 5):

- Στήλη 1 (Χαρακτηριστικό): Περιέχει διακριτές, ακέραιες τιμές. Κυμαίνονται από 2 έως 6, με μέση τιμή το 4. (Φαίνεται να πρόκειται για κάποια κατηγορική ή διαβαθμισμένη μεταβλητή, ίσως κάποια ρύθμιση εξοπλισμού, ύψος κεραίας κ.λπ.).

- Στήλη 2 (Χαρακτηριστικό): Ακέραιες τιμές που κυμαίνονται από το 871 έως το 1366, με μέση τιμή περίπου το 1119.

- Στήλη 3 (Χαρακτηριστικό): Ακέραιες τιμές που κυμαίνονται από το 744 έως το 1239, με μέση τιμή περίπου το 997. (Οι στήλες 2 και 3 φαίνονται να έχουν παρόμοια κλίμακα, ίσως συντεταγμένες x,y σε ένα πλέγμα ή κάποιες αποστάσεις).

- Στήλη 4 (Χαρακτηριστικό): Δεκαδικές τιμές από 5.02 έως 353.58. Η μέση τιμή είναι γύρω στο 192.23. Αυτό θα μπορούσε να είναι η απόσταση του πομπού από τον δέκτη (π.χ. σε μέτρα).

Όπως μου ανέφερες νωρίτερα, η 5η και τελευταία στήλη είναι το Path Loss (Απώλεια διαδρομής σε dB).

- Όλες οι τιμές είναι αρνητικές, πράγμα που είναι φυσιολογικό στις τηλεπικοινωνίες όταν μιλάμε για σήμα λήψης (ή path loss εκφρασμένο ως κέρδος).

- Εύρος Τιμών: Η ελάχιστη τιμή (μεγαλύτερη απώλεια) είναι -165.79 dB και η μέγιστη τιμή (μικρότερη απώλεια) είναι -52.95 dB.

- Μέση τιμή: Βρίσκεται στα -124.58 dB, ενώ το 50% των μετρήσεων (διάμεσος) είναι κοντά στο -128.66 dB.

Η 4η στήλη (οι δεκαδικοί αριθμοί που είδαμε πριν) είναι με τεράστια διαφορά το πιο σημαντικό στοιχείο του dataset σου!

Έχει πολύ ισχυρή αρνητική συσχέτιση (-0.69) με το Path Loss. Επειδή το Path Loss είναι αρνητικός αριθμός (π.χ. -120 dB), αυτό πρακτικά σημαίνει: Όσο αυξάνεται η τιμή της Στήλης 4, τόσο πιο αρνητικό γίνεται το Path Loss (δηλαδή τόσο μεγαλύτερη είναι η απώλεια σήματος). Αυτή είναι η κλασική συμπεριφορά της απόστασης (Distance) στις τηλεπικοινωνίες!

Το μοντέλο τής δίνει 78.6% βαρύτητα για τη λήψη αποφάσεων

Η Στήλη 3 έχει σημασία περίπου 13% για το μοντέλο.

Η Στήλη 2 έχει σημασία περίπου 8.3%.

Λογικά, αυτές οι δύο στήλες αντιπροσωπεύουν συντεταγμένες στον χώρο (X, Y) ή κάποια άλλα γεωγραφικά δεδομένα, τα οποία βοηθούν τον αλγόριθμο να καταλάβει αν υπάρχει κάποιο εμπόδιο σε μια συγκεκριμένη περιοχή.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# 1. Φόρτωση του dataset (βάζουμε header=None γιατί το αρχείο έχει κατευθείαν δεδομένα)
df = pd.read_excel('paper.xlsx', header=None)

# Ονομάζουμε τις στήλες για να είναι πιο κατανοητός ο κώδικας
df.columns = ['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Path_Loss']

# 2. Διαχωρισμός σε Χαρακτηριστικά (X) και Στόχο (y)
# Αν θες να δοκιμάσεις χωρίς τη Στήλη 1 (επειδή είχε μηδενική επιρροή), άλλαξε την παρακάτω γραμμή σε: X = df[['Feature_2', 'Feature_3', 'Feature_4']]
X = df.drop('Path_Loss', axis=1) 
y = df['Path_Loss']

# 3. Διαχωρισμός σε δεδομένα Εκπαίδευσης και Ελέγχου (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Πειραματισμός με διάφορες τιμές για το MaxDepth
max_depth_values = [2, 5, 10, 15, 20, 25, None] # None = το δέντρο μεγαλώνει ελεύθερα

# Επικεφαλίδες για την εκτύπωση των αποτελεσμάτων
print(f"{'MaxDepth':<10} | {'MAE':<8} | {'RMSE':<8} | {'MAPE':<8}")
print("-" * 45)

for depth in max_depth_values:
    # 5. Εκπαίδευση του Decision Tree
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    
    # 6. Πρόβλεψη πάνω στο Test set
    y_pred = model.predict(X_test)
    
    # 7. Υπολογισμός μετρικών (Σφάλματα)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Ρίζα του Mean Squared Error
    mape = mean_absolute_percentage_error(y_test, y_pred)
    
    # Εκτύπωση της γραμμής με τα αποτελέσματα
    depth_str = str(depth) if depth is not None else "None"
    print(f"{depth_str:<10} | {mae:<8.2f} | {rmse:<8.2f} | {mape:<8.4f}")

MaxDepth   | MAE      | RMSE     | MAPE    
---------------------------------------------
2          | 7.63     | 10.86    | 0.0667  
5          | 6.39     | 9.03     | 0.0549  
10         | 4.32     | 6.36     | 0.0359  
15         | 2.69     | 4.41     | 0.0219  
20         | 2.13     | 3.73     | 0.0172  
25         | 1.98     | 3.55     | 0.0159  
None       | 1.97     | 3.53     | 0.0158  


- Από Βάθος 2 σε Βάθος 10: Έχουμε τεράστια βελτίωση. Το σφάλμα (MAE) πέφτει σχεδόν στο μισό (από 7.63 σε 4.32).

- Από Βάθος 10 σε Βάθος 15: Έχουμε άλλη μια μεγάλη βουτιά στο σφάλμα (στο 2.69).

- Από Βάθος 15 και μετά (20, 25, None): Η βελτίωση είναι πλέον απειροελάχιστη (από 2.69 φτάνει μετά βίας στο 1.97).

Αν έπρεπε να βάλεις αυτό το μοντέλο σε παραγωγή (σε μια πραγματική εφαρμογή), το MaxDepth = 15 φαντάζει ως το "Sweet Spot" (το ιδανικό σημείο).

Στο βάθος 15, το μοντέλο σου έχει εξαιρετική ακρίβεια (μόλις ~2.69 dB απόκλιση στο Path Loss και ~2.1% μέσο ποσοστιαίο σφάλμα - MAPE).

MAE (Mean Absolute Error) - Μέσο Απόλυτο Σφάλμα

- Τι δείχνει: Κατά μέσο όρο, πόσο μακριά ήταν η πρόβλεψή μας από την πραγματική τιμή (ανεξάρτητα αν προβλέψαμε παραπάνω ή παρακάτω).

- Πώς υπολογίζεται: Το μοντέλο συγκρίνει την πρόβλεψή του (π.χ. -125 dB) με την πραγματική τιμή (π.χ. -120 dB). Το σφάλμα εδώ είναι 5 dB. Κάνει το ίδιο για όλα τα δείγματα στο test set και βγάζει τον μέσο όρο.

- Στο παράδειγμά σου: Όταν είδαμε MAE = 2.69, αυτό σημαίνει ότι κατά μέσο όρο, η πρόβλεψη του αλγορίθμου σου για την απώλεια σήματος έπεσε έξω κατά 2.69 dB.

Πλεονέκτημα: Είναι ο πιο εύκολος δείκτης για να καταλάβεις τι συμβαίνει, γιατί μετριέται στις ίδιες ακριβώς μονάδες με τα δεδομένα σου (σε dB).

RMSE (Root Mean Squared Error) - Ρίζα Μέσου Τετραγωνικού Σφάλματος

- Τι δείχνει: Και αυτός ο δείκτης μετράει το σφάλμα σε dB, αλλά "τιμωρεί" πολύ πιο αυστηρά τις πολύ κακές προβλέψεις (τα μεγάλα λάθη/outliers).

- Πώς υπολογίζεται: Υπολογίζει πρώτα τα σφάλματα, τα υψώνει στο τετράγωνο (έτσι τα μεγάλα νούμερα γίνονται τεράστια), βρίσκει τον μέσο όρο τους (MSE), και στο τέλος βγάζει τη ρίζα για να τα επαναφέρει στην αρχική κλίμακα των dB.

- Γιατί το χρειαζόμαστε; Φαντάσου ότι στο 90% των περιπτώσεων το μοντέλο σου πέφτει έξω κατά 1 dB, αλλά στο 10% πέφτει έξω κατά 50 dB! Το MAE μπορεί να φαίνεται χαμηλό, αλλά το RMSE θα εκτοξευτεί, "χτυπώντας καμπανάκι" ότι κάποιες προβλέψεις είναι τραγικές.

Στο παράδειγμά σου: Το RMSE (4.41 στο βάθος 15) είναι πάντα μεγαλύτερο (ή ίσο) από το MAE. Όσο πιο κοντά είναι το RMSE στο MAE, τόσο λιγότερα "ακραία" λάθη κάνει το μοντέλο σου.

MAPE (Mean Absolute Percentage Error) - Μέσο Απόλυτο Ποσοστιαίο Σφάλμα

- Τι δείχνει: Δείχνει το σφάλμα όχι σε απόλυτα νούμερα (dB), αλλά σε ποσοστό (%) επί της πραγματικής τιμής.

- Πώς υπολογίζεται: Αν η πραγματική απώλεια είναι 100 dB και το μοντέλο προέβλεψε 105 dB, το σφάλμα είναι 5%. Υπολογίζει αυτό το ποσοστό για κάθε δείγμα και βρίσκει τον μέσο όρο.

- Στο παράδειγμά σου: Είδαμε MAPE = 0.0219. Αν το πολλαπλασιάσεις επί 100 για να το κάνεις ποσοστό, προκύπτει 2.19%. Αυτό σημαίνει ότι, κατά μέσο όρο, οι προβλέψεις του αλγορίθμου σου αποκλίνουν κατά περίπου 2.2% από την πραγματική απώλεια του σήματος.

Εκπαιδεύστε SVR με: γραμμικό πυρήνα, RBF πυρήνα. Πειραματιστείτε με την παράμετρο:BoxConstraint (C) Υπολογίστε  MAE,RMSE, MAPE στο test set.

Γραμμικό (Linear) και τον Μη-Γραμμικό (RBF - Radial Basis Function). Για την παράμετρο C (BoxConstraint), δοκίμασα τιμές από 0.1 έως 100. Το C καθορίζει πόσο αυστηρά "τιμωρούμε" το μοντέλο όταν κάνει λάθη (μεγάλο C = προσπαθεί να μάθει τέλεια τα δεδομένα εκπαίδευσης).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

# 1. Φόρτωση και ονομασία στηλών
df = pd.read_excel('paper.xlsx', header=None)
df.columns = ['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Path_Loss']

X = df.drop('Path_Loss', axis=1)
y = df['Path_Loss']

# 2. Διαχωρισμός Δεδομένων (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. KΑΝΟΝΙΚΟΠΟΙΗΣΗ (SCALING) - Απαραίτητο για το SVR
scaler = StandardScaler()
# Το μοντέλο μαθαίνει την κλίμακα μόνο από το Train set...
X_train_scaled = scaler.fit_transform(X_train)
# ...και εφαρμόζει την ίδια κλίμακα στο Test set
X_test_scaled = scaler.transform(X_test)

# 4. Πειραματισμός με Kernels και BoxConstraint (C)
kernels = ['linear', 'rbf']
C_values = [0.1, 1, 10, 50, 100]

print(f"{'Kernel':<10} | {'C':<5} | {'MAE':<8} | {'RMSE':<8} | {'MAPE':<8}")
print("-" * 50)

for kernel in kernels:
    for c in C_values:
        # Δημιουργία και Εκπαίδευση του SVR
        model = SVR(kernel=kernel, C=c)
        model.fit(X_train_scaled, y_train)
        
        # Πρόβλεψη και Υπολογισμός Σφαλμάτων
        y_pred = model.predict(X_test_scaled)
        
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mape = mean_absolute_percentage_error(y_test, y_pred)
        
        print(f"{kernel:<10} | {c:<5} | {mae:<8.2f} | {rmse:<8.2f} | {mape:<8.4f}")# <8.2f σημαίνει ότι θα εκτυπωθεί με 2 δεκαδικά και θα καταλάβει τουλάχιστον 8 χαρακτήρες πλάτος για να ευθυγραμμιστεί σωστά στην εκτύπωση.

Kernel     | C     | MAE      | RMSE     | MAPE    
--------------------------------------------------
linear     | 0.1   | 8.38     | 12.63    | 0.0821  
linear     | 1     | 8.38     | 12.60    | 0.0820  
linear     | 10    | 8.38     | 12.59    | 0.0820  
linear     | 50    | 8.38     | 12.59    | 0.0820  
linear     | 100   | 8.38     | 12.59    | 0.0820  
rbf        | 0.1   | 7.81     | 11.51    | 0.0754  
rbf        | 1     | 6.71     | 9.45     | 0.0582  
rbf        | 10    | 6.48     | 9.19     | 0.0567  
rbf        | 50    | 6.31     | 9.08     | 0.0553  
rbf        | 100   | 6.25     | 9.05     | 0.0549  


Γραμμικός vs RBF Πυρήνας: Ο RBF νικάει κατά κράτος! Ο γραμμικός πυρήνας "κολλάει" στο MAE ~8.38 dB, ό,τι και αν κάνουμε με την παράμετρο C. Αυτό συμβαίνει γιατί υποθέτει ότι η σχέση των χαρακτηριστικών με το Path Loss είναι μια απλή ευθεία γραμμή.

Η πραγματικότητα (η διάδοση του σήματος) είναι μη-γραμμικό φαινόμενο, γι' αυτό ο RBF (που μπορεί να σχεδιάσει καμπύλες) τα πηγαίνει καλύτερα.

- Στον Γραμμικό, η αλλαγή του C δεν έχει σχεδόν καμία επίδραση.

- Στον RBF, όσο μεγαλώνουμε το C (π.χ. από 0.1 σε 100), τόσο βελτιώνεται το μοντέλο, ρίχνοντας το MAE από το 7.81 στο 6.25.

Tο Decision Tree (με MaxDepth=15) είχε MAE περίπου 2.69 dB. Το καλύτερο SVR εδώ έφτασε στα 6.25 dB. Για αυτό το συγκεκριμένο dataset, τα Δέντρα Απόφασης φαίνεται να κατανοούν τη γεωμετρία των δεδομένων σου πολύ καλύτερα από ό,τι τα Support Vector Machines!